In [2]:
!pip install -q transformers peft bitsandbytes accelerate trl datasets rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 49.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 53.7 MB/s eta 0:00:00


In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA available: True
Device: Tesla T4


In [4]:
pip install -U transformers huggingface_hub accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 102.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.8 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.3
    Uninstalling hf-xet-1.4.3:
      Successfully uninstalled hf-xet-1.4.3
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
  Attempting uninstall: huggingfa

In [5]:
!rm -rf /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-4B-Instruct-2507
Memory footprint: 2.59 GB


In [7]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # rank of the low-rank matrices
    lora_alpha=32,           # scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [8]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files="/kaggle/input/datasets/ballubalwan/llm-fine-tuning/training_data.jsonl", split="train")
eval_dataset = load_dataset("json", data_files="/kaggle/input/datasets/ballubalwan/llm-fine-tuning/eval_data.jsonl", split="train")

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")
print(f"\nSample:\n{train_dataset[0]['text'][:300]}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train examples: 1064
Eval examples: 119

Sample:
<|user|>
Write a research report answering: Encapsulating Security Payload (ESP) belongs to which Internet Security Protocol?
<|assistant|>
# Research Report: Encapsulating Security Payload (ESP) belongs to which Internet Security Protocol?

## Findings

The correct answer is: Secure IP Protocol. Th


In [9]:
import trl, transformers
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)

trl: 1.10.0
transformers: 5.15.0


In [10]:
import peft, accelerate
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

peft: 0.19.1
accelerate: 1.14.0


In [11]:
print(lora_config.target_modules)

{'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'k_proj', 'q_proj', 'v_proj'}


In [12]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen3-autolab-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    per_device_eval_batch_size=4,
    bf16=True,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
    loss_type="nll" # <-- opt out of the new chunked_nll default
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Trainer configured. Starting training...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1064 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1064 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1064 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1064 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1064 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainer configured. Starting training...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.235773,1.135659,1.104869,0.750045,227365.000000
2,1.183825,1.119978,1.063619,0.753225,454730.000000
3,1.092786,1.131269,0.990782,0.751796,682095.000000


TrainOutput(global_step=201, training_loss=1.1789063476211397, metrics={'train_runtime': 15074.2566, 'train_samples_per_second': 0.212, 'train_steps_per_second': 0.013, 'total_flos': 2.39743453676544e+16, 'train_loss': 1.1789063476211397, 'epoch': 3.0})

In [13]:
model.save_pretrained("/kaggle/working/qwen3-4b-autolab-lora/final_adapter")
tokenizer.save_pretrained("/kaggle/working/qwen3-4b-autolab-lora/final_adapter")
print("Adapter saved.")

Adapter saved.


In [14]:
!ls -la /kaggle/working/qwen3-4b-autolab-lora/final_adapter

total 75764
drwxr-xr-x 2 root root     4096 Aug 16 17:39 .
drwxr-xr-x 3 root root     4096 Aug 16 17:39 ..
-rw-r--r-- 1 root root     1106 Aug 16 17:39 adapter_config.json
-rw------- 1 root root 66127776 Aug 16 17:39 adapter_model.safetensors
-rw-r--r-- 1 root root     2630 Aug 16 17:39 chat_template.jinja
-rw-r--r-- 1 root root     5220 Aug 16 17:39 README.md
-rw-r--r-- 1 root root      695 Aug 16 17:39 tokenizer_config.json
-rw-r--r-- 1 root root 11422650 Aug 16 17:39 tokenizer.json


In [17]:
from rouge_score import rouge_scorer
import json
import re
import os

with open("/kaggle/input/datasets/ballubalwan/llm-fine-tuning/eval_data.jsonl") as f:  # adjust path if uploaded via Kaggle Datasets input
    eval_raw = [json.loads(line) for line in f]

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
RESULTS_PATH = "/kaggle/working/qwen3-4b-autolab-lora/eval_results.jsonl"

already_done = set()
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        for line in f:
            already_done.add(json.loads(line)["index"])
    print(f"Resuming: {len(already_done)} examples already evaluated.")

def generate_report(instruction, max_new_tokens=300):
    prompt = f"<|user|>\n{instruction}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def has_correct_structure(text):
    return "## Findings" in text and "## Sources" in text and bool(re.search(r"\[\d+\]", text))

with open(RESULTS_PATH, "a") as out_f:
    for i, ex in enumerate(eval_raw):
        if i in already_done:
            continue
        text = ex["text"]
        instruction = text.split("<|user|>\n")[1].split("\n<|assistant|>")[0]
        reference = text.split("<|assistant|>\n")[1]
        generated = generate_report(instruction)
        score = scorer.score(reference, generated)["rougeL"].fmeasure
        structure_ok = has_correct_structure(generated)
        result = {"index": i, "rouge_l": score, "structure_ok": structure_ok}
        out_f.write(json.dumps(result) + "\n")
        out_f.flush()
        if (i + 1) % 10 == 0:
            print(f"  Evaluated {i + 1}/{len(eval_raw)}...")

print("Evaluation complete.")

  Evaluated 10/119...
  Evaluated 20/119...
  Evaluated 30/119...
  Evaluated 40/119...
  Evaluated 50/119...
  Evaluated 60/119...
  Evaluated 70/119...
  Evaluated 80/119...
  Evaluated 90/119...
  Evaluated 100/119...
  Evaluated 110/119...
Evaluation complete.
